# EDA — CICIDS2017 depurado (pre-OE1: estabilidad del espacio latente)

**Contexto:** monografía *"Caracterización de la Geometría del Espacio Latente
Probabilístico de un VAE para la Detección de Intrusiones"*. Este notebook es el EDA de
preparación, previo a **OE1** (Semanas 1–2 del cronograma), sobre el dataset CICIDS2017
depurado ([Engelen, Rimmer & Joosen, 2021](https://intrusion-detection.distrinet-research.be/WTMC2021/)).

Complementa a `00_raw_data_eda.ipynb` (exploración cruda general) con foco puesto en las
decisiones que alimentan directamente el pipeline de preprocesamiento
(`vae_nids.data.pipeline`) y el diseño del encoder/decoder (arquitectura actual:
Dense-32 → latente-8, x ∈ R^78, ver §6.2 de la monografía):

1. Carga de datos
2. Perfilado básico (nulos/infinitos, distribución de clases, desbalance benigno/malicioso)
3. Chequeo de fuga de datos (Engelen et al.)
4. Distribuciones de features y matriz de correlación
5. Preparación para partición 70/15/15 + escalado sin fuga (funciones listas, **sin ejecutar** el split aún)
6. Resumen de hallazgos

**Kernel:** mismo `.venv` del proyecto (`vae_nids` está instalado en modo editable, así
que `import vae_nids` funciona directo).

**Nota de diseño:** donde el pipeline de producción (`src/vae_nids/data/pipeline.py`) ya
resuelve algo — taxonomía de labels, split estratificado, escalado sin fuga — este
notebook **importa y reutiliza esas mismas funciones** en vez de duplicar la lógica, para
que EDA y pipeline nunca diverjan (este mismo preprocesamiento se reutiliza en OE1–OE4).

In [ ]:
from pathlib import Path
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from vae_nids import config as cfg
from vae_nids.data import pipeline as dp

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

np.random.seed(cfg.RANDOM_SEED)

## 1. Carga de datos

Ajusta `N_ROWS_PER_DAY` a un entero (p. ej. `300_000`) si tu máquina no aguanta cargar
los ~1.1 GB completos de una sola vez.

A diferencia de `pipeline.load_and_merge` (que descarta `Flow ID`/IPs/`Timestamp`/puertos
y castea a `float32` al vuelo para mantener bajo el pico de memoria), aquí conservamos
**todas** las columnas porque la Sección 3 (chequeo de fuga) necesita poder inspeccionarlas
antes de decidir excluirlas.

In [ ]:
def load_dataset(nrows_per_day: int | None = None) -> pd.DataFrame:
    """Carga y concatena los 5 CSV crudos de `cfg.DATA_DIR`, conservando TODAS las
    columnas (a diferencia de `pipeline.load_and_merge`, que descarta identificadores
    y puertos al vuelo). Pensada para EDA; el entrenamiento real usa `pipeline.main()`.
    """
    frames = []
    for fname in cfg.CSV_FILES:
        fpath = cfg.DATA_DIR / fname
        if not fpath.exists():
            raise FileNotFoundError(
                f"No encuentro {fpath}. Ajusta cfg.DATA_DIR si tu copia vive en otro lugar."
            )
        day = fname.split("-")[0]
        df_day = pd.read_csv(fpath, nrows=nrows_per_day, low_memory=False)
        df_day.columns = df_day.columns.str.strip()
        df_day["Day"] = day
        frames.append(df_day)
        print(f"[load] {fname:30s} -> {len(df_day):>10,} filas")

    df = pd.concat(frames, ignore_index=True)
    del frames
    print(f"\n[load] total: {len(df):,} filas x {df.shape[1]} columnas")
    return df


N_ROWS_PER_DAY = None  # None = cargar todo; o un entero para muestrear rápido
df = load_dataset(N_ROWS_PER_DAY)

In [ ]:
print("shape:", df.shape)
df.dtypes.value_counts()

In [ ]:
list(df.columns)

## 2. Perfilado básico

### 2.1 Nulos e infinitos

In [ ]:
def count_nulls_infs(df: pd.DataFrame, cols=None) -> pd.DataFrame:
    """Cuenta nulos e infinitos por columna numérica. Relevante: Engelen et al. (2021)
    reportan artefactos de división por cero en tasas de paquetes/tiempo (p. ej.
    `Flow Bytes/s`, `Flow Packets/s`, `Flow IAT *`).
    """
    cols = cols if cols is not None else df.select_dtypes(include=[np.number]).columns
    n_inf = np.isinf(df[cols]).sum()
    n_null = df[cols].isna().sum()
    out = pd.DataFrame({"n_inf": n_inf, "n_null": n_null})
    out["pct_affected"] = (out["n_inf"] + out["n_null"]) / len(df)
    return out[(out["n_inf"] > 0) | (out["n_null"] > 0)].sort_values(
        "pct_affected", ascending=False
    )


numeric_cols = df.select_dtypes(include=[np.number]).columns
null_inf_report = count_nulls_infs(df, numeric_cols)
null_inf_report

Reutilizamos `pipeline.sanitize` (en vez de reimplementar el filtro) para medir de una
vez cuántas filas se perderían en el paso de sanitización real.

In [ ]:
_sanitized_preview = dp.sanitize(df.copy())
n_dropped = len(df) - len(_sanitized_preview)
print(f"[sanitize] filas descartadas: {n_dropped:,} ({n_dropped / len(df):.4%})")
del _sanitized_preview

### 2.2 Distribución de clases: benigno vs. familia de ataque

In [ ]:
df = dp.build_label_taxonomy(df)  # agrega is_benign / is_attempted / attack_family

label_counts = df[cfg.LABEL_COL].value_counts()
family_counts = df["attack_family"].value_counts()

class_table = pd.DataFrame({
    "count": family_counts,
    "pct_%": (family_counts / len(df) * 100).round(4),
})
class_table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
family_counts.sort_values().plot(kind="barh", ax=ax, logx=True)
ax.set_xlabel("count (escala log)")
ax.set_title("Distribución por familia de ataque (BENIGN vs. cada familia)")
plt.tight_layout()
plt.show()

### 2.3 Desbalance benigno/malicioso

In [ ]:
n_benign = int(df["is_benign"].sum())
n_attack = int((~df["is_benign"]).sum())
print(f"Benigno:    {n_benign:>10,} ({n_benign / len(df):.2%})")
print(f"Malicioso:  {n_attack:>10,} ({n_attack / len(df):.2%})")
print(f"Ratio benigno:malicioso = {n_benign / n_attack:.1f} : 1")

## 3. Chequeo de fuga de datos (Engelen et al., 2021)

Engelen et al. documentan que CICFlowMeter deja artefactos de la infraestructura de
simulación (TTL fijo, patrones de temporización artificiales) que un modelo puede
memorizar en vez de aprender comportamiento de ataque genuino. Dos chequeos:

1. Columnas constantes o cuasi-constantes → candidatas a ese tipo de artefacto.
2. Columnas identificadoras (IP/puerto/timestamp) que ya se excluyen por diseño en
   `cfg.IDENTIFIER_COLS` / `cfg.PORT_COLS` — aquí verificamos *por qué* tiene sentido
   excluirlas.

In [ ]:
def find_constant_columns(
    df: pd.DataFrame, cols=None, near_constant_thresh: float = 0.99
) -> pd.DataFrame:
    """Detecta columnas constantes o cuasi-constantes: candidatas a artefactos de
    simulación (TTL fijo, banderas siempre en el mismo estado, etc.).
    `near_constant_thresh`: si el valor más frecuente cubre >= este umbral, se marca
    como cuasi-constante.
    """
    cols = cols if cols is not None else df.columns
    rows = []
    for c in cols:
        vc = df[c].value_counts(normalize=True, dropna=False)
        top_val, top_frac = vc.index[0], float(vc.iloc[0])
        rows.append({
            "column": c,
            "n_unique": df[c].nunique(dropna=False),
            "top_value": top_val,
            "top_value_frac": top_frac,
            "constant": df[c].nunique(dropna=False) == 1,
            "near_constant": top_frac >= near_constant_thresh,
        })
    out = pd.DataFrame(rows).set_index("column")
    return out[out["constant"] | out["near_constant"]].sort_values(
        "top_value_frac", ascending=False
    )


non_feature_cols = ["Day", cfg.LABEL_COL, "is_benign", "is_attempted", "attack_family"]
feature_candidates = [c for c in df.columns if c not in non_feature_cols]

constant_report = find_constant_columns(df, feature_candidates, near_constant_thresh=0.99)
constant_report

In [ ]:
print("Columnas excluidas por diseño (fuga de identidad, ver cfg.IDENTIFIER_COLS / cfg.PORT_COLS):")
for c in cfg.IDENTIFIER_COLS + cfg.PORT_COLS:
    if c in df.columns:
        print(f"  {c:12s} {df[c].nunique():>10,} valores únicos")

Chequeo puntual: si las IPs de atacante/víctima son ~fijas por familia (como documenta
el README), cada familia de ataque debería tener muy pocas IPs origen distintas — lo que
confirma que dejarlas como feature sería fuga de identidad, no señal de comportamiento.

In [ ]:
if "Src IP" in df.columns:
    ip_by_family = (
        df.loc[~df["is_benign"]]
        .groupby("attack_family")["Src IP"]
        .nunique()
        .sort_values(ascending=False)
    )
    print("IPs origen únicas por familia de ataque (bajo == IP ~fija -> justifica excluirla):")
    ip_by_family

## 4. Distribuciones de features

### 4.1 Histogramas benigno vs. malicioso

In [ ]:
def plot_hist_by_class(
    df: pd.DataFrame,
    col: str,
    class_col: str = "is_benign",
    bins: int = 60,
    clip_quantile: float = 0.995,
    ax=None,
):
    """Histograma de `col` separado por clase (benigno/malicioso por defecto).
    Recorta al percentil `clip_quantile` para que outliers extremos (típicos en
    bytes/s, duración de flujo) no aplasten la escala visual.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))
    upper = df[col].quantile(clip_quantile)
    data = df[[col, class_col]].copy()
    data[col] = data[col].clip(upper=upper)
    sns.histplot(
        data=data, x=col, hue=class_col, bins=bins,
        stat="density", common_norm=False, ax=ax,
    )
    ax.set_title(f"{col} (recortado al percentil {clip_quantile:.1%})")
    return ax


# Requiere inf/nan ya fuera (ver Sección 2.1) para no distorsionar cuantiles/histogramas.
df_clean = dp.sanitize(df.copy())

key_features = [
    "Flow Duration",
    "Total Length of Fwd Packet",
    "Average Packet Size",
    "Flow Bytes/s",
    "Flow Packets/s",
]
key_features = [c for c in key_features if c in df_clean.columns]

fig, axes = plt.subplots(len(key_features), 1, figsize=(8, 4 * len(key_features)))
for ax, col in zip(np.atleast_1d(axes), key_features):
    plot_hist_by_class(df_clean, col, ax=ax)
plt.tight_layout()
plt.show()

### 4.2 Matriz de correlación

Se usa `pipeline.get_feature_columns` (el mismo selector que usará el pipeline real)
para operar exactamente sobre las columnas que verá el encoder, en vez de recalcular la
lista a mano.

In [ ]:
feature_cols = dp.get_feature_columns(df_clean)
corr = df_clean[feature_cols].corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, cbar_kws={"shrink": 0.6})
ax.set_title("Matriz de correlación — features numéricas")
plt.tight_layout()
plt.show()

In [ ]:
def find_correlated_pairs(corr: pd.DataFrame, threshold: float = 0.9) -> pd.DataFrame:
    """Pares de features con |correlación| >= threshold: redundancia candidata a
    recorte antes de fijar la dimensión de entrada del encoder (actualmente R^78).
    """
    mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
    pairs = (
        corr.where(mask)
        .stack()
        .rename("corr")
        .reset_index()
        .rename(columns={"level_0": "feature_a", "level_1": "feature_b"})
    )
    pairs = pairs[pairs["corr"].abs() >= threshold]
    return pairs.sort_values("corr", key=np.abs, ascending=False).reset_index(drop=True)


high_corr_pairs = find_correlated_pairs(corr, threshold=0.9)
print(f"{len(high_corr_pairs)} pares con |r| >= 0.9 (de {len(feature_cols)} features, "
      f"{len(feature_cols) * (len(feature_cols) - 1) // 2} pares posibles)")
high_corr_pairs

## 5. Preparación para partición (funciones listas, sin ejecutar)

La partición estratificada 70/15/15 sobre el subconjunto benigno y el `MinMaxScaler`
ajustado solo con el split de entrenamiento (§6.3 de la monografía) **ya están
implementados** en `src/vae_nids/data/pipeline.py` como `split_benign` y
`scale_no_leakage` — no se reimplementan aquí para evitar que EDA y pipeline diverjan.

Esta sección solo confirma que las funciones están listas e importables; **no se ejecuta
el split sobre el dataset completo en este notebook** (eso corresponde a
`python -m vae_nids.data.pipeline`, ver README, una vez cerradas las decisiones de la
Sección 6 — p. ej. qué columnas cuasi-constantes o redundantes excluir antes de fijar la
lista definitiva de features).

In [ ]:
print(inspect.getsource(dp.split_benign))

In [ ]:
print(inspect.getsource(dp.scale_no_leakage))

In [ ]:
print("cfg.TRAIN_FRAC / VAL_FRAC / TEST_FRAC:", cfg.TRAIN_FRAC, cfg.VAL_FRAC, cfg.TEST_FRAC)
print("cfg.STRATIFY_BENIGN_BY_DAY:", cfg.STRATIFY_BENIGN_BY_DAY)
print("cfg.RANDOM_SEED:", cfg.RANDOM_SEED)
print()
print("Para generar train/val/test/attacks + scaler.joblib, ejecutar desde la raíz del repo:")
print("    python -m vae_nids.data.pipeline")

## 6. Resumen de hallazgos

_(completar a mano tras correr las secciones de arriba sobre el dataset completo)_

- **% de filas descartadas en sanitización** (Sección 2.1): `<completar>` — comparar
  contra el ~0.04% documentado en el README para el dataset depurado.
- **Features candidatas a eliminar** (Sección 3, constantes/cuasi-constantes): `<listar
  columnas de `constant_report`>`.
- **Features redundantes** (Sección 4.2, `high_corr_pairs`): `<listar pares con |r| ≥ 0.9
  y decidir cuáles recortar>`.
- **Desbalance benigno/malicioso** (Sección 2.3): `<completar ratio>` — confirma la
  necesidad del paradigma one-class descrito en la Sección 1 de la monografía.
- **Hallazgos que afectan el diseño del encoder/decoder** (arquitectura actual: Entrada
  x ∈ R^78 → Dense-32 → latente-8 → Dense-32 → salida ∈ R^78): `<completar — p. ej. si
  hay features altamente redundantes, ¿vale la pena recortar antes de OE1, o dejar que el
  encoder de 32 unidades absorba la redundancia?>`.
- **Otras anomalías / decisiones pendientes para OE1:** `<completar>`.